In [4]:
!pip install gradio

   ---------------------------------------- 0.0/31.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/31.3 MB ? eta -:--:--
   ---------------------------------------- 0.3/31.3 MB ? eta -:--:--
    --------------------------------------- 0.5/31.3 MB 839.5 kB/s eta 0:00:37
    --------------------------------------- 0.5/31.3 MB 839.5 kB/s eta 0:00:37
    --------------------------------------- 0.5/31.3 MB 839.5 kB/s eta 0:00:37
    --------------------------------------- 0.5/31.3 MB 839.5 kB/s eta 0:00:37
    --------------------------------------- 0.5/31.3 MB 839.5 kB/s eta 0:00:37
    --------------------------------------- 0.5/31.3 MB 839.5 kB/s eta 0:00:37
    --------------------------------------- 0.5/31.3 MB 839.5 kB/s eta 0:00:37
    --------------------------------------- 0.5/31.3 MB 839.5 kB/s eta 0:00:37
    --------------------------------------- 0.5/31.3 MB 839.5 kB/s eta 0:00:37
    --------------------------------------- 0.5/31.3 MB 839.5 kB/s eta 0:00:37


In [5]:
import gradio as gr
import joblib
import faiss
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [7]:
classifier = joblib.load("ministry_model.pkl")

vectorizer = joblib.load("tfidf.pkl")

index = faiss.read_index("parliament.index")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

df = pd.read_csv("my_data_q.csv")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
def predict(text):

    vector = vectorizer.transform([text])

    prediction = classifier.predict(vector)

    return prediction[0]

def semantic_search(query):

    vector = embedding_model.encode(
        [query]
    ).astype("float32")

    distance, indices = index.search(
        vector,
        5
    )

    results = df.iloc[
        indices[0]
    ][
        [
            "subjects",
            "ministry",
            "member"
        ]
    ]

    return results.to_markdown(index=False)


def recommend(query):

    return semantic_search(query)


In [9]:
def parliament_ai(query):

    ministry = predict(query)

    similar = semantic_search(query)

    recommendation = recommend(query)

    return ministry, similar, recommendation

    

In [22]:
demo = gr.Interface(

    fn=parliament_ai,

    inputs=gr.Textbox(

        lines=3,

        label="Ask ParliamentAI"

    ),

    outputs=[

        gr.Textbox(label="Predicted Ministry"),

        gr.Markdown(label="Similar Questions"),

        gr.Markdown(label="Recommendations")

    ],

    title="🇮🇳 ParliamentAI",

    description="""
    Intelligent Search, Analysis &
    Recommendation System
    """

)

In [23]:
demo.close()

In [24]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
